# 02. 손실함수와 경사하강법 — 직접 굴려보기

**대응 강의:** [3강 학습의 본질](../../Lecture/03-머신러닝-기본개념/03-학습의-본질.md)

'학습'이라는 말, 들으면 뭔가 추상적이죠? 이 노트북에서 그 정체를 코드로 하나씩 뜯어볼 거예요. 같이 해볼 건:
- 손실함수(MSE)가 그리는 **'골짜기' 모양**을 직접 그려 보기
- 경사하강법을 **라이브러리 없이 손으로 구현**해서, 한 걸음씩 바닥으로 내려가는 과정 보기
- **학습률(learning rate)** 을 바꿔가며 잘 수렴할 때 / 너무 느릴 때 / 발산할 때를 직접 만들어 보기

> 추상적이던 '학습'이 사실은 단순한 반복이라는 걸 느끼게 될 거예요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 간단한 가짜 데이터를 만들어요: 진짜 관계는 y = 3x (기울기 3)에 노이즈를 살짝 섞은 거예요
rng = np.random.default_rng(0)
x = np.linspace(-2, 2, 50)
true_w = 3.0
y = true_w * x + rng.normal(0, 1.0, size=x.shape)

plt.scatter(x, y, alpha=0.6)
plt.title('Data (hidden true slope = 3)')
plt.xlabel('x'); plt.ylabel('y'); plt.show()

## 1. 손실함수의 '골짜기' 그려 보기

모델을 아주 단순하게 `y = w * x` 로 두고, **파라미터 `w` 딱 하나**만 학습한다고 해볼게요.
`w`에 이 값 저 값 넣어가며 MSE 손실을 계산해 보면, 손실이 결국 `w`에 따라 달라지는 함수라는 게 보여요.

$$\text{MSE}(w) = \frac{1}{n}\sum_i (y_i - w\,x_i)^2$$

In [ ]:
def mse_loss(w):
    pred = w * x
    return np.mean((y - pred) ** 2)

w_grid = np.linspace(-2, 8, 200)
losses = [mse_loss(w) for w in w_grid]

plt.plot(w_grid, losses)
plt.axvline(true_w, color='g', ls='--', label='true w=3 (near valley bottom)')
plt.xlabel('parameter w'); plt.ylabel('loss (MSE)')
plt.title('The loss valley - we want to find its bottom')
plt.legend(); plt.show()

## 2. 경사하강법을 직접 구현해 보기

업데이트 규칙은 이거예요: $w \leftarrow w - \eta \cdot \nabla L(w)$

MSE의 기울기(그래디언트)를 직접 미분해서 구하면 이렇게 나와요:
$$\nabla L(w) = \frac{d}{dw}\,\frac{1}{n}\sum (y_i - w x_i)^2 = -\frac{2}{n}\sum x_i (y_i - w x_i)$$

In [ ]:
def gradient(w):
    pred = w * x
    return -2 * np.mean(x * (y - pred))   # MSE를 w로 미분한 값

def gradient_descent(lr=0.1, steps=30, w_start=-1.0):
    w = w_start
    history = [w]
    for _ in range(steps):
        w = w - lr * gradient(w)     # ★ 핵심은 이 한 줄: 기울기 반대 방향으로 한 걸음
        history.append(w)
    return np.array(history)

path = gradient_descent(lr=0.1, steps=30, w_start=-1.0)
print('시작 w =', round(path[0], 3))
print('최종 w =', round(path[-1], 3), ' (진짜 값 3에 수렴했어요!)')

In [ ]:
# 경사하강이 골짜기를 따라 내려가는 모습을 그려 봐요
plt.plot(w_grid, losses, alpha=0.5, label='loss valley')
plt.scatter(path, [mse_loss(w) for w in path], c=range(len(path)),
            cmap='autumn_r', s=40, zorder=3, label='gradient descent steps')
plt.plot(path, [mse_loss(w) for w in path], 'k-', alpha=0.3)
plt.xlabel('parameter w'); plt.ylabel('loss')
plt.title('Stepping down to the valley (brighter = later steps)')
plt.legend(); plt.show()

## 3. 학습률(η) 실험 — 제일 중요한 손잡이

코드는 그대로 두고 **학습률만** 바꿔 볼게요. 너무 작으면 답답할 만큼 느리고, 너무 크면 골짜기를 튕겨 나가 발산해 버려요.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
settings = [(0.005, 'too small: crawling'), (0.1, 'just right: stable'), (0.55, 'too big: diverges!')]

for ax, (lr, title) in zip(axes, settings):
    p = gradient_descent(lr=lr, steps=30, w_start=-1.0)
    p = np.clip(p, w_grid.min(), w_grid.max())   # 그래프 범위 안으로 잘라 줘요
    ax.plot(w_grid, losses, alpha=0.4)
    ax.plot(p, [mse_loss(w) for w in p], 'o-', ms=4)
    ax.set_title(f'lr={lr}\n{title}')
    ax.set_xlabel('w'); ax.set_ylabel('loss')
plt.tight_layout(); plt.show()

print('발산하는 경우 w가 어떻게 변하는지:', gradient_descent(lr=0.55, steps=8).round(2))

## 정리

- **손실함수**는 파라미터에 따라 값이 변하는 함수이고, 그 모양이 '골짜기'예요.
- **경사하강법**은 기울기 반대 방향으로 한 걸음씩 내려가는 거예요 = `w ← w - η·∇L`
- **학습률**이 너무 작으면 굼뜨고, 너무 크면 발산해요. 딱 적당한 값을 찾는 게 관건이에요.

> 💡 이 반복(예측 → 손실 → 기울기 → 이동)이 선형회귀부터 신경망까지, 거의 모든 학습의 뼈대예요.

## 🎯 직접 해보기

1. `lr`을 0.55와 0.1 사이(예: 0.3, 0.45)로 바꿔가며 '발산하기 직전'의 경계가 어디인지 찾아보세요.
2. `w_start`를 7.0(반대편 언덕)에서 시작해도 똑같은 바닥으로 수렴하나요?
3. (도전) 모델을 `y = w*x + b`로 바꿔서 파라미터를 **두 개**(w, b) 학습하도록 늘려 보세요. w, b 각각에 대해 기울기를 구하면 돼요.